# KRW Money Counter - YOLOv8n training

Kaggle setup: **Add Input** -> upload `krw-dataset.zip` as a dataset, **Accelerator** -> GPU T4/P100, **Internet** -> on.

Outputs in `/kaggle/working/out`: `best.pt`, `best.onnx`, `metrics.json`, plots.

In [ ]:
!pip install -q ultralytics==8.3.40

In [ ]:
import glob, json, os, shutil, yaml
from ultralytics import YOLO, settings

settings.update({'raytune': False})  # Kaggle's preinstalled Ray breaks this callback

src_yaml = glob.glob('/kaggle/input/**/data.yaml', recursive=True)[0]
root = os.path.dirname(src_yaml)
cfg = yaml.safe_load(open(src_yaml))
cfg['path'] = root
yaml.safe_dump(cfg, open('/kaggle/working/data.yaml', 'w'), sort_keys=False)
print(root, cfg['names'])

In [ ]:
model = YOLO('yolov8n.pt')  # COCO pre-trained weights, fine-tuned on our 8 classes
model.train(
    data='/kaggle/working/data.yaml',
    epochs=30, imgsz=640, batch=32, patience=8,
    fliplr=0.0,           # mirrored bills/coins don't exist in real life
    degrees=10, scale=0.5, hsv_v=0.4,
    project='/kaggle/working/runs', name='krw', exist_ok=True,
)

In [ ]:
best = YOLO('/kaggle/working/runs/krw/weights/best.pt')
m = best.val(data='/kaggle/working/data.yaml', split='test', imgsz=640, project='/kaggle/working/runs', name='test', exist_ok=True)

names = best.names
metrics = {
    'test_mAP50': round(float(m.box.map50), 4),
    'test_mAP50_95': round(float(m.box.map), 4),
    'test_precision': round(float(m.box.mp), 4),
    'test_recall': round(float(m.box.mr), 4),
    'per_class_mAP50': {names[int(c)]: round(float(v), 4) for c, v in zip(m.box.ap_class_index, m.box.ap50)},
    'inference_ms': round(float(m.speed['inference']), 2),
}
print(json.dumps(metrics, indent=2))

In [ ]:
onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True)

out = '/kaggle/working/out'
os.makedirs(out, exist_ok=True)
shutil.copy('/kaggle/working/runs/krw/weights/best.pt', out)
shutil.copy(onnx_path, out)
json.dump(metrics, open(f'{out}/metrics.json', 'w'), indent=2)
for f in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'val_batch0_pred.jpg']:
    p = f'/kaggle/working/runs/krw/{f}'
    if os.path.exists(p):
        shutil.copy(p, out)
shutil.copy('/kaggle/working/runs/test/confusion_matrix_normalized.png', f'{out}/test_confusion_matrix.png')
print(os.listdir(out))